In [ ]:
!pip -q install pyspark
from google.colab import files
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
spark = SparkSession.builder.appName("StreamingSet4").getOrCreate()
files.upload()  # Select all six CSV files
def read_csv(name):
    return spark.read.csv(name, header=True, inferSchema=True)
sessions_df = read_csv("watch_sessions_80000.csv")
extra_df = read_csv("watch_sessions_extra_10000.csv")
subscribers_df = read_csv("subscriber_master_2400.csv")
content_df = read_csv("content_catalog_120.csv")
devices_df = read_csv("device_master_8.csv")
plans_df = read_csv("subscription_plans_4.csv")
print("Main sessions:", sessions_df.count())
sessions_df.printSchema()
sessions_df.show(5, truncate=False)

**Part A - DataFrame operations**

In [ ]:
sessions_df.printSchema()
print(sessions_df.count())
sessions_df.show(10)

In [ ]:
#2
from pyspark.sql.functions import col
sessions_df.filter((col("session_status")=="Completed") & (col("watch_minutes")>60)).show()
sessions_df.select( "session_id", "subscriber_id", "content_id", "watch_minutes", "session_status").show()

In [ ]:
#3
sessions_df=(sessions_df.filter(col("watch_minutes")>60)
.withColumn("watch_hours",col("watch_minutes")/60)
.withColumn("engagement_flag",col("completion_pct")>=75))
sessions_df.select("watch_minutes","watch_hours","completion_pct","engagement_flag").show()

In [ ]:
#4
sessions_df.orderBy(col("data_used_mb").desc()).show(20)

**Part B - Data quality and conditional transformations**

In [ ]:
#5
print(
    sessions_df.filter(col("subscriber_id").isNull()).count(),
    sessions_df.filter(col("device_id").isNull()).count(),
    sessions_df.filter(col("completion_pct").isNull()).count(),
    sessions_df.filter(col("buffering_seconds").isNull()).count(),
    sessions_df.filter(col("user_rating").isNull()).count(),

)

In [ ]:
#6
new_df=sessions_df.na.fill({"buffering_seconds":0})
new_df.show(5)

In [ ]:
#7
new_df = sessions_df.filter(col("device_id").isNotNull())
print("Original count:", sessions_df.count())
print("New DataFrame count:", new_df.count())

In [ ]:
#8
from pyspark.sql.functions import when
sessions_df=sessions_df.withColumn("completion_sts",

                                   when(col("completion_pct").isNull(),"Unknown")
                                  .when(col("completion_pct")<30,"Low")
                                  .when(col("completion_pct")<90,"Medium")
                                  .otherwise("High"))
sessions_df.select("completion_pct","completion_sts").show()

**Part C - Joins and aggregations**

In [ ]:
content_df.show(1)
sessions_df.show(1)

In [ ]:
#9
from pyspark.sql.functions import count,sum,avg
session_content=sessions_df.join(content_df,on="content_id",how="inner")
(session_content.groupBy("genre")
.agg((sum("watch_minutes").alias("total_watch_minutes")),(count("*").alias("session_count"))
)).show()


In [ ]:
#10
session_device=sessions_df.join(devices_df,on="device_id",how="inner")
(session_device.groupBy("device_type")
.agg((avg("buffering_seconds").alias("Average_buff_seconds")),(avg("watch_minutes").alias("Avg_watch_minutes")))
).show()

In [ ]:
plans_df.show(1)
subscribers_df.show(1)
sessions_df.show(1)

In [ ]:
#11
session_subscribe_plan=(sessions_df
                        .join(subscribers_df,on="subscriber_id",how="inner")
                        .join(plans_df,on="plan_id",how="inner"))
(session_subscribe_plan.filter(col("subscriber_id").isNotNull())
 .groupBy("plan_name")
 .agg(
        (
            count(when(col("session_status") == "Completed", True)) /
            count("*") * 100
        ).alias("completion_rate")
)).show()

In [ ]:
#12
plan_subscriber=subscribers_df.join(plans_df,on="plan_id",how="inner")
(plan_subscriber.filter(col("subscriber_status")=="Active")
.groupBy("plan_id")
.agg(sum("monthly_fee_inr").alias("monthly_revenue"),count("*").alias("subscriber_count"))
).show()

In [ ]:
#13

unmatched_subscribers = sessions_df.join(
    subscribers_df,
    on="subscriber_id",
    how="left_anti"
).filter(col("subscriber_id").isNotNull())

unmatched_devices = sessions_df.join(
    devices_df,
    on="device_id",
    how="left_anti"
).filter(col("device_id").isNotNull())

print("Unmatched subscribers:", unmatched_subscribers.count())
print("Unmatched devices:", unmatched_devices.count())

**Part D - Date and window functions**

In [ ]:
#14
from pyspark.sql.functions import col, to_timestamp, date_format, count, sum

sessions_df = sessions_df.withColumn(
    "session_start",
    to_timestamp(col("session_start"))
)

(sessions_df
 .filter(col("session_start").isNotNull() & (col("session_start").substr(1, 4) == "2026"))
 .withColumn("month", date_format(col("session_start"), "yyyy-MM"))
 .groupBy("month")
 .agg(
     count("*").alias("session_count"),
     sum("watch_minutes").alias("total_watch_minutes")
 )
 .orderBy("month")
).show()

In [ ]:
#15
from pyspark.sql.functions import datediff, to_date

session_signup = sessions_df.join(
    subscribers_df.select("subscriber_id", "signup_date"),
    on="subscriber_id",
    how="inner"
)

(session_signup
 .withColumn(
     "days_since_signup",
     datediff(to_date(col("session_start")), to_date(col("signup_date")))
 )
 .orderBy(col("session_start").asc())
 .select("subscriber_id", "session_start", "signup_date", "days_since_signup")
 .show(10)
)

In [ ]:
#16
from pyspark.sql.functions import dayofweek, when, avg

sessions_df = sessions_df.withColumn(
    "day_type",
    when(dayofweek(col("session_start")).isin(1, 7), "Weekend")
    .otherwise("Weekday")
)

(sessions_df
 .groupBy("day_type")
 .agg(avg("watch_minutes").alias("average_watch_minutes"))
 .show()
)

In [ ]:
#17
from pyspark.sql.functions import dense_rank
from pyspark.sql.window import Window

session_content = sessions_df.join(
    content_df,
    on="content_id",
    how="inner"
)

genre_window = Window.partitionBy("genre").orderBy(
    col("total_watch_minutes").desc()
)

genre_totals = (session_content
    .groupBy("genre", "content_title")
    .agg(sum("watch_minutes").alias("total_watch_minutes"))
)

(genre_totals
 .withColumn("rank", dense_rank().over(genre_window))
 .filter(col("rank") <= 3)
 .orderBy("genre", "rank")
 .show()
)

In [ ]:
#18
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

subscriber_window = Window.partitionBy("subscriber_id").orderBy(
    col("session_start").desc(),
    col("session_id").desc()
)

(sessions_df
 .filter(col("subscriber_id").isNotNull())
 .withColumn("row_num", row_number().over(subscriber_window))
 .filter(col("row_num") == 1)
 .drop("row_num")
 .show()
)

In [ ]:
#19
from pyspark.sql.functions import lag

subscriber_window = Window.partitionBy("subscriber_id").orderBy(
    col("session_start"),
    col("session_id")
)

(sessions_df
 .filter(col("subscriber_id").isNotNull())
 .withColumn(
     "previous_watch_minutes",
     lag("watch_minutes").over(subscriber_window)
 )
 .withColumn(
     "watch_minutes_difference",
     col("watch_minutes") - col("previous_watch_minutes")
 )
 .select(
     "subscriber_id",
     "session_start",
     "session_id",
     "watch_minutes",
     "previous_watch_minutes",
     "watch_minutes_difference"
 )
 .show()
)

**Part E - Union, lazy execution, caching and partitioning**

In [ ]:
#20
combined_df = sessions_df.unionByName(
    extra_df,
    allowMissingColumns=True
)

print("Total records:", combined_df.count())

deduplicated_df = combined_df.dropDuplicates(["session_id"])

print("Records after removing duplicates:", deduplicated_df.count())

In [ ]:
#21
from pyspark.sql.functions import col, lit, sum

lazy_df = (sessions_df
    .filter(col("session_status") == "Completed")
    .join(content_df, on="content_id", how="inner")
    .withColumn("watch_hours", col("watch_minutes") / 60)
    .groupBy("genre")
    .agg(sum("watch_hours").alias("total_watch_hours"))
    .orderBy(col("total_watch_hours").desc())
)

lazy_df.explain()

In [ ]:
#22
from pyspark.sql.functions import count, avg
import time

completed_df = sessions_df.filter(
    col("session_status") == "Completed"
)

completed_df.cache()

start = time.time()
completed_df.count()
print("First count time:", time.time() - start)

start = time.time()
completed_df.agg(
    avg("watch_minutes").alias("avg_watch_minutes")
).show()
print("First aggregation time:", time.time() - start)

start = time.time()
completed_df.count()
print("Second count time:", time.time() - start)

start = time.time()
completed_df.agg(
    avg("watch_minutes").alias("avg_watch_minutes")
).show()
print("Second aggregation time:", time.time() - start)

completed_df.unpersist()

In [ ]:
#23
from pyspark import StorageLevel

four_k_df = sessions_df.filter(
    col("stream_quality") == "4K"
)

four_k_df.persist(StorageLevel.MEMORY_AND_DISK)

four_k_df.count()

four_k_df.agg(
    avg("watch_minutes").alias("avg_watch_minutes")
).show()

four_k_df.unpersist()

In [ ]:
#24
from pyspark.sql.functions import spark_partition_id, count

partitioned_df = sessions_df.repartition(8)

(partitioned_df
 .groupBy(spark_partition_id().alias("partition_id"))
 .agg(count("*").alias("record_count"))
 .orderBy("partition_id")
 .show()
)

In [ ]:
#25
import os
partitioned_df = sessions_df.repartition(8)

coalesce_df = partitioned_df.coalesce(2)

repartition_df = partitioned_df.repartition(2)

print("Coalesce execution plan:")
coalesce_df.explain()

print("Repartition execution plan:")
repartition_df.explain()

coalesce_df.write.mode("overwrite").parquet("/content/sessions_parquet")



print(os.listdir("/content/sessions_parquet"))

**Final capstone - Streaming performance report**

In [ ]:
#1
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel

combined_df = sessions_df.unionByName(
    extra_df,
    allowMissingColumns=True
)

print("Total records:", combined_df.count())

deduplicated_df = combined_df.dropDuplicates(["session_id"])

print("Records after deduplication:", deduplicated_df.count())

In [ ]:
#2
cleaned_df = (deduplicated_df
    .filter(F.col("session_id").isNotNull())
    .withColumn(
        "watch_minutes",
        F.coalesce(F.col("watch_minutes"), F.lit(0))
    )
    .withColumn(
        "buffering_seconds",
        F.coalesce(F.col("buffering_seconds"), F.lit(0))
    )
    .withColumn(
        "data_used_mb",
        F.coalesce(F.col("data_used_mb"), F.lit(0))
    )
)

In [ ]:
#3
subscriber_master = subscribers_df.select(
    "subscriber_id", "region", "plan_id"
)

content_master = content_df.select(
    "content_id", "content_title", "genre"
)

device_master = devices_df.select(
    "device_id", "device_type"
)

plan_master = plans_df.select(
    "plan_id", "plan_name"
)

joined_df = (cleaned_df
    .join(subscriber_master, on="subscriber_id", how="left")
    .join(content_master, on="content_id", how="left")
    .join(device_master, on="device_id", how="left")
    .join(plan_master, on="plan_id", how="left")
)

In [ ]:
#4
monthly_df = (joined_df
    .filter(col("session_start").isNotNull())
    .withColumn(
        "session_start",
        to_timestamp(col("session_start"))
    )
    .withColumn(
        "month",
        date_format(col("session_start"), "yyyy-MM")
    )
    .withColumn(
        "watch_hours",
        col("watch_minutes") / 60
    )
)

monthly_metrics_df = (monthly_df
    .groupBy("month", "region", "genre")
    .agg(
        count("*").alias("session_count"),

        sum("watch_hours").alias("total_watch_hours"),

        (
            avg(
                when(col("session_status") == "Completed", 1.0)
                .otherwise(0.0)
            ) * 100
        ).alias("completion_rate"),

        avg("buffering_seconds").alias("avg_buffering_seconds"),

        sum("data_used_mb").alias("total_data_used_mb")
    )
)

monthly_metrics_df.show()

In [ ]:
#5
title_totals_df = (monthly_df
    .groupBy("genre", "content_title")
    .agg(
        sum("watch_hours").alias("title_watch_hours")
    )
)

genre_window = Window.partitionBy("genre").orderBy(
    col("title_watch_hours").desc()
)

top_titles_df = (title_totals_df
    .withColumn(
        "rank",
        dense_rank().over(genre_window)
    )
    .filter(col("rank") <= 3)
    .orderBy("genre", "rank")
)

top_titles_df.show()

In [ ]:
#6
monthly_report_df = monthly_metrics_df.withColumn(
    "report_type", F.lit("Monthly Metrics")
)

title_report_df = top_titles_df.withColumn(
    "report_type", F.lit("Top Three Titles")
)

final_report_df = monthly_report_df.unionByName(
    title_report_df,
    allowMissingColumns=True
)

final_report_df.cache()


print("Total report records:", final_report_df.count())


final_report_df.show(20, truncate=False)


final_report_df.unpersist()

In [ ]:
#7
import os

final_report_df.repartition(3).write \
    .mode("overwrite") \
    .parquet("/content/streaming_performance_report")

print(os.listdir("/content/streaming_performance_report"))